In [18]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

In [19]:
dataset = pd.read_csv('chembl262_chembl2850_06_bioactivity_data_3class_pIC50_pubchem_fp.csv')
dataset

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,pIC50
0,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,8.301030
1,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.004365
2,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.795880
3,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.494850
4,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,8.154902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3560,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,9.267606
3561,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.380907
3562,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.000000
3563,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.000000


In [20]:
# if the dataset contains NaN values than apply this
dataset = dataset.dropna()


In [21]:
# Define the target variable Y
Y = dataset.iloc[:, -1]

In [22]:
# Remove low variance features
def remove_low_variance(input_data, threshold=0.1):
    selection = VarianceThreshold(threshold)
    selection.fit(input_data)
    return input_data[input_data.columns[selection.get_support(indices=True)]]

In [23]:
X = dataset.drop(['pIC50'], axis=1)
X = remove_low_variance(X, threshold=0.1)
X


,PubchemFP2,PubchemFP12,PubchemFP16,PubchemFP19,PubchemFP20,PubchemFP23,PubchemFP33,PubchemFP37,PubchemFP143,PubchemFP145,...,PubchemFP756,PubchemFP758,PubchemFP776,PubchemFP777,PubchemFP779,PubchemFP797,PubchemFP798,PubchemFP800,PubchemFP819,PubchemFP821
0,0,0,1,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,1,1,1,0,0,0,0,1,1,...,0,0,0,1,0,0,0,0,0,0
2,0,0,1,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,1,1,...,0,1,1,0,0,0,0,1,0,1
4,0,1,1,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3560,1,1,1,1,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3561,1,1,1,1,0,1,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3562,1,1,1,1,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3563,1,1,1,1,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [24]:
# Save the descriptor list
X.to_csv('descriptor_list.csv', index=False)

In [25]:
# Data split (80/20 ratio)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2021)


In [26]:
# Model Development Libararies (LGBRegressor)
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, explained_variance_score, mean_squared_log_error
from sklearn.ensemble import HistGradientBoostingRegressor as HGBRegressor
from sklearn.svm import NuSVR
from sklearn.ensemble import VotingRegressor
import pickle

In [27]:
from lightgbm import LGBMRegressor

# GridSearchCV

# Define the LGBMRegressor model
lgbm = LGBMRegressor(random_state=2021)

# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 40, 50],
    'max_depth': [5, 10, 15, -1],
    'min_child_samples': [20, 30, 50, 100],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0, 0.1, 0.5, 1.0]
}

# Set up RandomizedSearchCV
random_search_lgbm = RandomizedSearchCV(
    lgbm,
    param_distributions=param_dist,
    n_iter=50,  # Number of parameter settings that are sampled
    scoring='r2', # Use R-squared as the scoring metric
    cv=5,       # 5-fold cross-validation
    verbose=2,  # Output progress
    random_state=2021,
    n_jobs=-1   # Use all available CPU cores
)

# Fit RandomizedSearchCV to the training data
random_search_lgbm.fit(X_train, Y_train)

print("Best parameters found: ", random_search_lgbm.best_params_)
print("Best R-squared score: ", random_search_lgbm.best_score_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009960 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 482
[LightGBM] [Info] Number of data points in the train set: 2852, number of used features: 241
[LightGBM] [Info] Start training from score 6.190404
Best parameters found:  {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'num_leaves': 40, 'n_estimators': 200, 'min_child_samples': 30, 'max_depth': -1, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best R-squared score:  0.517035337095112


### Evaluate LGBM Regressor

In [28]:
# Make predictions on the test set for the LGBM model
Y_pred_lgbm = random_search_lgbm.best_estimator_.predict(X_test)

def evaluate_model(Y_true, Y_pred, model_name):
    print(f"\n--- {model_name} Evaluation ---")
    print(f"Mean Squared Error: {mean_squared_error(Y_true, Y_pred):.4f}")
    print(f"R2 Score: {r2_score(Y_true, Y_pred):.4f}")
    print(f"Mean Absolute Error: {mean_absolute_error(Y_true, Y_pred):.4f}")
    print(f"Explained Variance Score: {explained_variance_score(Y_true, Y_pred):.4f}")

# Evaluate the LGBMRegressor model
evaluate_model(Y_test, Y_pred_lgbm, "LightGBM Regressor")


--- LightGBM Regressor Evaluation ---
Mean Squared Error: 0.8985
R2 Score: 0.5027
Mean Absolute Error: 0.7158
Explained Variance Score: 0.5046


### Saving the LGBM Regressor Model

In [30]:
import pickle

# Save the best LGBM model using pickle
pickle.dump(random_search_lgbm.best_estimator_, open('LGBM_Regressor_Model.pkl', 'wb'))
print("LGBM Regressor model saved to 'LGBM_Regressor_Model.pkl'")

LGBM Regressor model saved to 'LGBM_Regressor_Model.pkl'
